# LLMOps, Monitoring, Cost & Reliability — Applied

**AI Architecture · Week 22b**

Offline notebook for operating the deployed insurance-underwriter assistant: SLO contract, synthetic weekly telemetry, dashboard evaluator, incident runbook selector, and eval regression gate. No network calls.

## 1. Customer SLO contract

Targets negotiated with underwriting, compliance, platform, and finance: availability ≥ 99.5%, p95 latency ≤ 6s, groundedness ≥ 92%, refusal ≤ 8%, hallucination ≤ 1%, cost/query ≤ $0.08, tool success ≥ 98%.

In [ ]:
from __future__ import annotations
from collections import defaultdict
from statistics import mean
from typing import Literal
import numpy as np
from pydantic import BaseModel, ConfigDict, Field

Status = Literal['PASS','AT_RISK','BREACH']
class CustomerSLOContract(BaseModel):
    model_config = ConfigDict(extra='forbid')
    availability: float = Field(0.995, ge=0, le=1)
    p95_latency_s: float = Field(6.0, gt=0)
    groundedness: float = Field(0.92, ge=0, le=1)
    refusal_rate: float = Field(0.08, ge=0, le=1)
    hallucination_rate: float = Field(0.01, ge=0, le=1)
    cost_per_query_usd: float = Field(0.08, gt=0)
    tool_call_success: float = Field(0.98, ge=0, le=1)
    monthly_availability_error_budget_min: float = 216.0

contract = CustomerSLOContract()
print(contract.model_dump())

In [ ]:
class RequestRecord(BaseModel):
    tenant: str
    latency_s: float
    answered: bool = True
    grounded_score: float
    refused: bool = False
    hallucinated: bool = False
    cost_usd: float
    tool_ok: bool = True
    query_embedding_shift: float = 0.0
    available: bool = True

class SLOResult(BaseModel):
    observed: float
    target: float
    status: Status
    delta: float

class PeriodReport(BaseModel):
    slo_results: dict[str, SLOResult]
    error_budget_remaining_pct: float
    top_cost_tenants: list[tuple[str, float]]
    top_drift_indicator: str
    markdown: str

def p95(values):
    values = sorted(values)
    return values[int(round(0.95 * (len(values)-1)))] if values else 0.0

def classify(observed, target, higher_is_better):
    margin = observed - target if higher_is_better else target - observed
    if margin >= 0: return 'PASS'
    return 'AT_RISK' if margin >= -abs(target)*0.05 else 'BREACH'

def evaluate_period(records, contract=contract):
    total = len(records); answered = [r for r in records if r.answered]
    values = {
        'availability': (sum(r.available for r in records)/total, contract.availability, True),
        'p95_latency_s': (p95([r.latency_s for r in answered]), contract.p95_latency_s, False),
        'groundedness': (mean([r.grounded_score for r in answered]), contract.groundedness, True),
        'refusal_rate': (sum(r.refused for r in records)/total, contract.refusal_rate, False),
        'hallucination_rate': (sum(r.hallucinated for r in records)/total, contract.hallucination_rate, False),
        'cost_per_query_usd': (mean([r.cost_usd for r in answered]), contract.cost_per_query_usd, False),
        'tool_call_success': (sum(r.tool_ok for r in records)/total, contract.tool_call_success, True),
    }
    results = {k: SLOResult(observed=round(v,4), target=t, status=classify(v,t,h), delta=round(v-t,4)) for k,(v,t,h) in values.items()}
    availability = values['availability'][0]
    downtime_min = (1-availability) * contract.monthly_availability_error_budget_min / (1-contract.availability)
    remaining = max(0.0, 100.0 * (1 - downtime_min/contract.monthly_availability_error_budget_min))
    spend = defaultdict(float); drift = defaultdict(list)
    for r in records:
        spend[r.tenant] += r.cost_usd; drift[r.tenant].append(r.query_embedding_shift)
    top_cost = sorted(spend.items(), key=lambda kv: kv[1], reverse=True)[:3]
    top_drift = max(((t, mean(v)) for t, v in drift.items()), key=lambda kv: kv[1])
    lines = ['# Weekly Underwriter AI SLO Report','','| SLO | Observed | Target | Status | Delta |','|---|---:|---:|---|---:|']
    for name, res in results.items(): lines.append(f'| {name} | {res.observed:.4f} | {res.target:.4f} | {res.status} | {res.delta:+.4f} |')
    lines += ['', f'Error budget remaining: **{remaining:.1f}%**', 'Top cost tenants:']
    lines += [f'- {t}: ${c:.2f}' for t,c in top_cost]
    lines.append(f'Top drift indicator: {top_drift[0]} shift {top_drift[1]:.3f}')
    return PeriodReport(slo_results=results, error_budget_remaining_pct=round(remaining,1), top_cost_tenants=[(t, round(c,2)) for t,c in top_cost], top_drift_indicator=f'{top_drift[0]}:{top_drift[1]:.3f}', markdown='\n'.join(lines))

## 2. Generate deterministic normal and bad weeks

In [ ]:
def synth_week(seed: int, bad: bool = False) -> list[RequestRecord]:
    rng = np.random.default_rng(seed)
    tenants = np.array(['commercial-lines','west-region','specialty','excess'])
    records = []
    for i in range(50):
        tenant = str(tenants[i % len(tenants)])
        latency = float(rng.normal(4.6 if not bad else 5.8, 0.45))
        grounded = float(rng.normal(0.935 if not bad else 0.902, 0.018))
        cost = float(rng.normal(0.047 if not bad else 0.073, 0.008))
        if bad and tenant == 'west-region': cost += 0.055
        records.append(RequestRecord(
            tenant=tenant, latency_s=max(1.0, latency), grounded_score=max(0.0, min(1.0, grounded)),
            refused=bool(rng.random() < (0.05 if not bad else 0.10)), hallucinated=bool(rng.random() < (0.004 if not bad else 0.025)),
            cost_usd=max(0.005, cost), tool_ok=bool(rng.random() > (0.01 if not bad else 0.04)),
            query_embedding_shift=float(rng.normal(0.10 if not bad else (0.28 if tenant == 'west-region' else 0.14), 0.03)),
            available=bool(rng.random() > (0.002 if not bad else 0.012))))
    return records

normal_week = synth_week(22, bad=False)
bad_week = synth_week(23, bad=True)
print(len(normal_week), len(bad_week), normal_week[0].model_dump())

## 3. Evaluate both weeks and print customer reports

In [ ]:
for label, records in [('NORMAL WEEK', normal_week), ('BAD WEEK', bad_week)]:
    print('\n========', label, '========')
    report = evaluate_period(records)
    print(report.markdown)

## 4. Incident runbook selector

In [ ]:
SignalType = Literal['groundedness_drop','cost_spike','latency_regression','safety_flag_spike','provider_5xx_burst','injection_pattern_detected']
Axis = Literal['prompt','model','index','tenant_limit','provider_route','guardrail']
class AlertPayload(BaseModel):
    model_config = ConfigDict(extra='forbid')
    signal_type: SignalType
    magnitude: float
    tenant: str
    model: str
    prompt_version: str
class Runbook(BaseModel):
    incident: str
    sli_to_check_first: str
    rollback_axis: Axis
    steps: list[str]
    customer_message_template: str
    post_incident_correction: str

def select_runbook(alert):
    common = ['Acknowledge page', 'Attach App Insights traces and release tuple']
    if alert.signal_type == 'groundedness_drop':
        return Runbook(incident='2am groundedness canary drop', rollback_axis='model', sli_to_check_first='hourly golden-set groundedness delta', steps=common+['Verify eval agent health','Rollback model deployment or prompt pointer','Open Azure provider case','Watch next canary'], customer_message_template='We detected and mitigated a controlled-canary quality regression for {tenant}; user traffic is protected while validation continues.', post_incident_correction='Add failed canaries to golden set and pin/isolate model deployment.')
    if alert.signal_type == 'cost_spike':
        return Runbook(incident='Runaway context cost spike', rollback_axis='tenant_limit', sli_to_check_first='cost/query and prompt-token p95 by tenant/prompt', steps=common+['Slice by tenant, feature, prompt, model','Apply temporary tenant rate limit','Rollback prompt/index if release-caused','Review usage with customer owner'], customer_message_template='We isolated a spend anomaly for {tenant}, applied a temporary budget guard, and are reviewing the usage pattern with your team.', post_incident_correction='Add cost regression gate and top-k/context-token alert.')
    if alert.signal_type == 'injection_pattern_detected':
        return Runbook(incident='Suspected prompt injection', rollback_axis='guardrail', sli_to_check_first='safety flags and audit report of prompt leakage', steps=common+['Capture Blob audit record','Block similar pattern','Escalate to security','Pause affected prompt/tenant route if needed'], customer_message_template='We detected a suspected prompt-injection pattern for {tenant}, contained similar prompts, preserved evidence, and escalated to security.', post_incident_correction='Add attack to red-team evals and strengthen delimiters/precedence.')
    return Runbook(incident='Generic AI operations alert', rollback_axis='prompt', sli_to_check_first='SLO dashboard', steps=common+['Triage release tuple','Mitigate safest rollback axis'], customer_message_template='We are investigating an AI operations alert for {tenant}.', post_incident_correction='Add representative case to eval set.')

In [ ]:
alerts = [
    AlertPayload(signal_type='groundedness_drop', magnitude=0.025, tenant='commercial-lines', model='gpt-4o-prod', prompt_version='prompt-v21'),
    AlertPayload(signal_type='cost_spike', magnitude=4.0, tenant='west-region', model='gpt-4o-prod', prompt_version='prompt-v21'),
    AlertPayload(signal_type='injection_pattern_detected', magnitude=1.0, tenant='commercial-lines', model='gpt-4o-prod', prompt_version='prompt-v21'),
]
for alert in alerts:
    rb = select_runbook(alert)
    print('\n##', rb.incident)
    print('first SLI:', rb.sli_to_check_first)
    print('rollback axis:', rb.rollback_axis)
    for step in rb.steps: print('-', step)
    print('customer:', rb.customer_message_template.format(tenant=alert.tenant))
    print('correction:', rb.post_incident_correction)

## 5. Eval regression gate for a prompt change

In [ ]:
golden = [
    {'id':'g1','topic':'regulatory filing','baseline_grounded':0.91,'candidate_grounded':0.94,'baseline_cost':0.050,'candidate_cost':0.052},
    {'id':'g2','topic':'precedent memo','baseline_grounded':0.92,'candidate_grounded':0.95,'baseline_cost':0.047,'candidate_cost':0.049},
    {'id':'g3','topic':'binding authority edge case','baseline_grounded':0.94,'candidate_grounded':0.88,'baseline_cost':0.051,'candidate_cost':0.064},
]
def eval_regression_gate(rows, max_grounded_drop_pp=1.0, max_cost_delta_pct=10.0):
    base_g = mean(r['baseline_grounded'] for r in rows); cand_g = mean(r['candidate_grounded'] for r in rows)
    base_c = mean(r['baseline_cost'] for r in rows); cand_c = mean(r['candidate_cost'] for r in rows)
    grounded_delta_pp = (cand_g - base_g) * 100
    cost_delta_pct = (cand_c - base_c) / base_c * 100
    bad_cases = [r['id'] for r in rows if (r['candidate_grounded'] - r['baseline_grounded']) * 100 < -max_grounded_drop_pp]
    decision = 'PASS' if grounded_delta_pp >= -max_grounded_drop_pp and cost_delta_pct <= max_cost_delta_pct and not bad_cases else 'BLOCK'
    return decision, round(grounded_delta_pp, 2), round(cost_delta_pct, 1), bad_cases
print('Prompt PR gate:', eval_regression_gate(golden))
for row in golden:
    print(row['id'], row['topic'], 'grounded delta pp', round((row['candidate_grounded']-row['baseline_grounded'])*100, 2))

## Exercises

1. Add tenant-specific SLO overrides for a high-risk underwriting unit.
2. Add a synthetic provider 5xx incident and route it through the runbook selector.
3. Change cache hit rate from 18% to 25% and estimate monthly generation savings.
4. Add one stale-label example to the golden set and show how it can create a false alert.

## Links
- Literature note: `02 Literature Notes/AI Architecture/LLMOps, Monitoring, Cost & Reliability — Applied`
- Snippets: `04 Code Snippets/AI Architecture/AI Week 22b Underwriter SLO Dashboard Evaluator`, `.../AI Week 22b AI Incident Response Classifier and Runbook Selector`
- MOC: `06 Maps of Content/AI Architecture Concepts`